# Lab 0-03: One LLM, an API Request, and Structured Output

Use this tutorial before `03_model_comparison.ipynb`. You will use the one model configured in this lab's `.env` twice on the same synthetic forensic task. First, you will inspect a normal chat response. Then, you will ask for a response that follows a JSON Schema.

The goal is to understand the request-and-response pattern before comparing three different models. Structured output makes a response easier for a program to read because the program knows which fields to expect.

## Step 0: Load This Lab's Settings

Run this cell from the `lab0_03_model_basics` folder. It reads the configured `MODEL` and `OLLAMA_BASE_URL`, then creates the same OpenAI-compatible client used in the later labs.

In [ ]:
# `json` reads the model's structured response; `Path` finds this lab's files.
import json
from pathlib import Path

from dotenv import dotenv_values
from openai import OpenAI

# Confirm that Jupyter was opened from this lab, not from the repository root.
LAB_NAME = 'lab0_03_model_basics'
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

# The private `.env` selects the model and local Ollama address for this lab.
env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

# Read settings without printing their full contents.
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')
if not model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

# Ollama accepts this OpenAI-compatible client; its local API key value is ignored.
client = OpenAI(base_url=ollama_base_url, api_key='ollama')
print('Model:', model)
print('Ollama address:', ollama_base_url)

## Step 1: Ask One Ordinary API Question

A chat-completions request has a model name and a list of messages. This first request uses no output constraint, so the model is free to choose the wording and layout of its answer.

### Understanding `messages`

`messages=[{'role': 'user', 'content': task_prompt}]` is the conversation sent to the model. The square brackets mean **a list of messages**, so later requests can include more than one turn of conversation. Each message is a dictionary with two important parts:

- `'role': 'user'` identifies who sent the message. The `user` role is the person or program making the request: it gives the model the immediate task, question, or evidence to work with. In this notebook, the user message asks the model to read the case note and extract information.
- A `system` message is different: it sets higher-level behavior for the conversation, such as the model's role, allowed scope, safety boundary, or required writing style. For example, a system message might say, `You are a forensic assistant. Use only the supplied case note and say when information is unknown.` It does not replace the user's task; it provides the rules the model should follow while answering that task.
- An `assistant` message represents an earlier model response. Including one lets a later request continue a conversation or ask the model to revise its own earlier answer.
- `'content': task_prompt` supplies the actual text of the request. Here, `task_prompt` is a variable containing both the instructions and the synthetic case note.

For example, a three-message conversation could look like this:

```python
messages = [
    {'role': 'system', 'content': 'You are a forensic assistant. Use only the supplied case note and say when information is unknown.'},
    {'role': 'user', 'content': 'Read this case note and identify the investigator.'},
    {'role': 'assistant', 'content': 'The investigator is Maya Chen.'},
]
```

Here, the `system` message sets the evidence boundary, the `user` message asks for a specific task, and the `assistant` message records the model's earlier answer. A later user message could ask the model to check or revise that answer.

For this first request, the list has only one user message: *read this case note and identify the requested information*. The model reads that message and returns its answer in `response.choices[0].message.content`.

In [ ]:
# Keep one small synthetic note so both requests use identical evidence.
case_note = '''
Investigator Maya Chen documented an interview with Jordan Lee.
Jordan said a suspicious text came from 415-555-0187.
The seized phone record listed IMEI 356938035643809.
'''.strip()

# This prompt describes the facts to extract. It is reused for both requests.
task_prompt = f'''
Read this synthetic case note. Identify the investigator and the suspicious phone number.
Give a brief case summary.

Case note:
{case_note}
'''.strip()

# Send a normal chat request. Without `response_format`, the model chooses its layout.
plain_response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': task_prompt}],
)
# The OpenAI-compatible response stores the assistant's text in the first choice.
plain_text = plain_response.choices[0].message.content
print(plain_text)

The answer may be useful to a person, but its layout is chosen by the model. A program that needs exact fields would need to guess where each fact appears.

## Step 2: Define the Output Shape

A JSON Schema describes the fields and value types the program expects. The schema below requires a short summary plus two extraction fields.

In [ ]:
# A JSON Schema tells the API which object fields and value types we expect.
extraction_schema = {
    'type': 'object',
    'properties': {
        'case_summary': {'type': 'string'},
        'investigator': {'type': 'string'},
        'suspicious_phone_number': {'type': 'string'},
    },
    'required': [
        'case_summary', 'investigator', 'suspicious_phone_number',
    ],
    # Do not allow unplanned fields; this keeps later program steps predictable.
    'additionalProperties': False,
}

print(json.dumps(extraction_schema, indent=2))

## Step 3: Request Structured Output

This request uses the same model and case note, but adds `response_format`. Keep the prompt specific as well: the schema controls the shape, while the prompt tells the model what the values should mean.

In [ ]:
# Reuse the same model and prompt, but ask the API to enforce `extraction_schema`.
structured_response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': task_prompt}],
    response_format={
        'type': 'json_schema',
        'json_schema': {
            # The name labels this response format; `strict` enforces the schema.
            'name': 'forensic_case_extraction',
            'strict': True,
            'schema': extraction_schema,
        },
    },
)
# The API still returns text, so the next step turns JSON text into a Python dictionary.
structured_text = structured_response.choices[0].message.content
print(structured_text)

## Step 4: Parse and Check the Result

JSON text is still text until your program parses it. This check confirms that every required field is present and contains a string. In a larger application, the same checks would protect later steps that depend on the extraction.

In [ ]:
# Parse the JSON string before code can access individual extraction fields.
structured_data = json.loads(structured_text)
required_fields = extraction_schema['required']
# Check for missing fields, incorrect value types, and fields outside the schema.
missing_fields = [field for field in required_fields if field not in structured_data]
non_string_fields = [
    field for field in required_fields
    if field in structured_data and not isinstance(structured_data[field], str)
]
unexpected_fields = sorted(set(structured_data) - set(extraction_schema['properties']))

# Stop with a useful report if the returned data is not safe for later code to use.
if missing_fields or non_string_fields or unexpected_fields:
    raise ValueError({
        'missing_fields': missing_fields,
        'non_string_fields': non_string_fields,
        'unexpected_fields': unexpected_fields,
    })

print('Structured output passed the checks.')
print(json.dumps(structured_data, indent=2))

## Step 5: Short Exercise—Add One Field

Add a required `recommended_next_step` string field to both `properties` and `required`. Then update `task_prompt` so the model recommends one human review step based only on the case note. Rerun Steps 3 and 4.

Your result is complete when the parsed output includes `recommended_next_step` and the validation check still passes.

## Before You Continue

You have now used one LLM through an API and turned a model response into a predictable data object. Next, open `03_model_comparison.ipynb`, where three models will receive the same prompt and you will compare their outputs.